
# Phân tích đánh giá sản phẩm & Sentiment Analysis

# 1. Giới thiệu và mục tiêu

Mục tiêu của dự án là **phân tích đánh giá sản phẩm và thực hiện sentiment analysis** trên các review của khách hàng.

Dựa vào nội dung đánh giá, ta xác định được mức độ **hài lòng, không hài lòng hoặc trung lập**, từ đó rút ra insight giúp doanh nghiệp cải thiện chất lượng sản phẩm và trải nghiệm người dùng.

### Nhiệm vụ cụ thể
- **Thu thập dữ liệu**: Đọc và xử lý dữ liệu review từ `merged_reviews.json`
- **Check**: kiểm tra nội dung content của các đánh giá với rating của đánh giá đã phù hợp chưa
- **Làm sạch dữ liệu**: Xử lý văn bản tiếng Việt, tách từ, chuẩn hóa cụm phủ định
- **Tạo nhãn**: Phân loại sentiment thành 3 lớp: **negative (1-2 sao)**, **neutral (3 sao)**, **positive (4-5 sao)**
- **Khám phá dữ liệu**: Phân tích phân phối rating, sentiment, độ dài review, từ phổ biến
- **Xây dựng mô hình**: Sử dụng **TF-IDF + LogisticRegression** cho bài toán phân loại 3 lớp
- **Đánh giá**: Đo lường hiệu suất mô hình và đưa ra đề xuất cải thiện



In [ ]:
# Import thư viện cần thiết cho phân tích dữ liệu

import re
import json
import warnings
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Set, Callable

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Cấu hình logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler('sentiment_analysis.log', encoding='utf-8')
    ]
)
logger = logging.getLogger(__name__)

plt.rcParams['font.family'] = 'DejaVu Sans'
pd.set_option('display.max_colwidth', 150)

# Đường dẫn dữ liệu
DATA_PATH = 'data-review/merged_reviews.json'

# Cấu hình
USE_UNDERTHESEA = True
STOPWORDS_PATH = None
MIN_WORDS = 5

logger.info(f"Đường dẫn dữ liệu: {DATA_PATH}")
logger.info(f"Cấu hình: USE_UNDERTHESEA={USE_UNDERTHESEA}, MIN_WORDS={MIN_WORDS}")

try:
    from underthesea import word_tokenize as uts_word_tokenize 
    _UNDER_THE_SEA_AVAILABLE = True
    logger.info("Underthesea đã được cài đặt và sẵn sàng sử dụng")
except Exception as e:
    _UNDER_THE_SEA_AVAILABLE = False
    logger.warning(f"Underthesea không khả dụng: {e}")



# 2. Thu thập và làm sạch dữ liệu

### Quy trình xử lý dữ liệu

### 2.1. Thu thập dữ liệu (dữ liệu đã được thu thập thông qua script)
- Đọc dữ liệu từ file `merged_reviews.json`
- Kiểm tra cấu trúc và chất lượng dữ liệu ban đầu
- Xử lý các vấn đề về encoding và format

### 2.2. Làm sạch dữ liệu
- **Ép kiểu thời gian**: Chuyển đổi cột `created_at` sang datetime
- **Tạo nhãn sentiment**: Chuyển đổi rating thành 3 lớp sentiment
- **Ghép văn bản**: Kết hợp `title + content` (loại bỏ giá trị `None`)
- **Tiền xử lý văn bản**: 
  - Loại bỏ URL, email, emoji
  - Chuẩn hóa ký tự tiếng Việt
  - Tách từ bằng underthesea
  - Ghép cụm phủ định (không_hài_lòng, rất_không_hài_lòng)
- **Lọc dữ liệu**: Loại bỏ review ngắn (< 5 từ) và trùng lặp

### 2.3. Kết quả xử lý
- So sánh kích thước dữ liệu trước và sau làm sạch
- Hiển thị mẫu dữ liệu đã xử lý


In [ ]:
# === 2.2. Làm sạch dữ liệu ===

# Thiết lập tham số và đường dẫn dữ liệu
#Define các hàm để xử lý dữ liệu

# Stopwords bổ sung để loại bỏ từ không có ý nghĩa
ADDITIONAL_STOPWORDS = {
    # -- Thương hiệu / số / nền tảng --
    'tiki', 'shopee', 'lazada', 'sendo', 'shop', 'store', 'website', 'app',
    'web', 'site', 'page', 'sp', 'san', 'hang', 'cua', 'hanghoa',
    '1', '2', '3', '4', '5', '6', '7', '8', '9', '0',

    # -- Đại từ chỉ định --
    'này', 'kia', 'đó', 'ấy', 'đây', 'đấy', 'nọ', 'nớ', 'ấy', 'đấy',

    # -- Đại từ nhân xưng (ít mang nghĩa trong sentiment) --
    'tôi', 'tao', 'tớ', 'mình', 'bạn', 'cậu', 'anh', 'chị', 'em', 'ông', 'bà', 
    'cô', 'chú', 'bác', 'họ', 'nó', 'ai', 'người', 'chúng', 'chúng tôi', 'chúng ta',

    # -- Giới từ / liên từ / hư từ --
    'và', 'với', 'hoặc', 'nhưng', 'mà', 'thì', 'là', 'bởi', 'vì', 'nếu', 'để',
    'do', 'như', 'kiểu', 'bằng', 'qua', 'từ', 'đến', 'tới', 'trong', 'ngoài', 
    'trên', 'dưới', 'giữa', 'sau', 'trước', 'tại', 'bên', 'khi', 'lúc', 'sau đó',

    # -- Từ nối và trợ từ --
    'vậy', 'thế', 'thì', 'lại', 'đều', 'cũng', 'chỉ', 'mới', 'nữa', 'đã', 'rồi', 'cả',
    'đang', 'được', 'bị', 'cần', 'phải', 'nên', 'hay', 'thôi', 'nhé', 'nhá', 'nha', 'à', 'ạ', 'ơ', 'ừ',

    # -- Trạng từ thời gian (ít giá trị phân loại cảm xúc) --
    'nay', 'mai', 'qua', 'hôm', 'giờ', 'lúc', 'ngày', 'tháng', 'năm', 'sáng', 'trưa', 'chiều', 'tối', 'đêm',

    # -- Định lượng và chỉ số lượng --
    'nhiều', 'ít', 'vài', 'một', 'hai', 'ba', 'bốn', 'năm', 'mấy', 'toàn', 'hết', 'cả', 'mọi', 'mỗi', 'bất kỳ', 'từng',

    # -- Từ cảm thán hoặc đệm không mang nghĩa --
    'ha', 'hà', 'hihi', 'haha', 'huhu', 'uh', 'ừ', 'ờ', 'ơ', 'ok', 'okay', 'okie', 'oki', 'okela',
    'ơi', 'nè', 'nha', 'nhé', 'hả', 'há', 'à', 'ạ', 'ơ', 'hừ', 'ừm',

    # -- Các từ chỉ nơi chốn không mang cảm xúc --
    'đây', 'kia', 'đó', 'này', 'đấy', 'kia', 'đằng', 'phía', 'chỗ', 'nơi', 'vị trí',

    # -- Dạng viết tắt hoặc sai chính tả thường gặp --
    'sp', 'sanpham', 'spnày', 'spnay', 'hanghoa', 'hanghoa', 'spnè', 'sảnphẩm', 'sanpham', 'san pham'
}

def _load_stopwords(path: str | None) -> Set[str]:
    if not path:
        return set()
    p = Path(path)
    if not p.exists():
        warnings.warn(f"Stopwords file not found: {path}")
        return set()
    words: Set[str] = set()
    for line in p.read_text(encoding='utf-8', errors='ignore').splitlines():
        w = line.strip()
        if not w or w.startswith('#'):
            continue
        words.add(w)
    return words

_URL_RE = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
_EMAIL_RE = re.compile(r"[\w\.-]+@[\w\.-]+\.[a-zA-Z]{2,}")
_EMOJI_RE = re.compile(r"[\U00010000-\U0010FFFF]", flags=re.UNICODE)

def clean_text_basic(text: str) -> str:
    if text is None:
        return ""
    s = str(text)
    if s.strip().lower() == 'none':
        s = ''
    s = _URL_RE.sub(' ', s)
    s = _EMAIL_RE.sub(' ', s)
    s = _EMOJI_RE.sub(' ', s)
    s = re.sub(r'[^\w\sáàạảãăắằặẳẵâấầậẩẫéèẹẻẽêếềệểễíìịỉĩóòọỏõôốồộổỗơớờợởỡúùụủũưứừựửữýỳỵỷỹđ]', ' ', s)
    s = re.sub(r"\s+", ' ', s).strip()
    s = s.lower()
    return s

def _join_negation_phrases(tokens: list[str]) -> list[str]:
    if not tokens:
        return tokens
    out = []
    i = 0
    while i < len(tokens):
        # Kiểm tra cụm 4 từ: "rất không hài lòng"
        if i + 3 < len(tokens) and tokens[i:i+4] == ['rất','không','hài','lòng']:
            out.append('rất_không_hài_lòng'); i += 4; continue
            
        # Kiểm tra cụm 3 từ: "không hài lòng"
        if i + 2 < len(tokens) and tokens[i:i+3] == ['không','hài','lòng']:
            out.append('không_hài_lòng'); i += 3; continue
            
        # Kiểm tra cụm 3 từ với hài_lòng đã được ghép sẵn: "rất không hài_lòng"
        if i + 2 < len(tokens) and tokens[i:i+3] == ['rất','không','hài_lòng']:
            out.append('rất_không_hài_lòng'); i += 3; continue
            
        # Kiểm tra cụm 2 từ với hài_lòng đã được ghép sẵn
        if i + 1 < len(tokens):
            bi = tokens[i:i+2]
            if bi == ['rất','không']:
                out.append('rất_không'); i += 2; continue
            if bi == ['không','hài_lòng']:
                out.append('không_hài_lòng'); i += 2; continue
            if bi == ['rất','hài_lòng']:
                out.append('rất_hài_lòng'); i += 2; continue
            if bi == ['chưa','tốt']:
                out.append('chưa_tốt'); i += 2; continue
            if bi == ['tạm','được']:
                out.append('tạm_được'); i += 2; continue
                
        out.append(tokens[i]); i += 1
    return out

def basic_tokenize(s: str) -> list[str]:
    s = re.sub(r"\s+", ' ', s).strip()
    return s.split()

def make_vn_tokenizer(use_underthesea: bool, stopwords: Set[str]) -> Callable[[str], list[str]]:
    def _tokenize(doc: str) -> list[str]:
        cleaned = clean_text_basic(doc)
        if use_underthesea and _UNDER_THE_SEA_AVAILABLE:
            # Underthesea có thể tách "rất không hài lòng" thành ["rất", "không", "hài", "lòng"]
            # nên cần áp dụng logic ghép cụm từ ngay sau khi tách từ
            toks = uts_word_tokenize(cleaned, format='text').split()
        else:
            toks = basic_tokenize(cleaned)
        
        # Áp dụng logic ghép cụm từ ngay sau khi tách từ
        toks = _join_negation_phrases(toks)
        
        # Kết hợp stopwords từ file và stopwords bổ sung
        all_stopwords = stopwords.union(ADDITIONAL_STOPWORDS) if stopwords else ADDITIONAL_STOPWORDS
        toks = [t for t in toks if t not in all_stopwords and t != 'none']
        return toks
    return _tokenize

def label_from_rating_3class(r):
    try:
        r = int(r)
    except Exception:
        return None
    if r in [1, 2]:
        return 'negative'
    if r == 3:
        return 'neutral'
    if r in [4, 5]:
        return 'positive'
    return None

def safe_join_text(row: pd.Series, cols: list[str]) -> str:
    parts: list[str] = []
    for c in cols:
        if c in row and pd.notna(row[c]) and str(row[c]).strip().lower() != 'none':
            parts.append(str(row[c]).strip())
    return ' '.join(parts).strip()

def _clean_control_chars(text: str) -> str:
    return ''.join(ch for ch in text if (ch >= ' ' or ch in '\n\r\t'))

def load_reviews(path: str) -> pd.DataFrame:
    logger.info(f"Bắt đầu tải dữ liệu từ: {path}")
    p = Path(path)
    if not p.exists():
        logger.error(f"File không tồn tại: {path}")
        raise FileNotFoundError(f"Không tìm thấy file: {path}")
    
    logger.info("Đọc file và làm sạch ký tự control...")
    raw = p.read_text(encoding='utf-8', errors='ignore')
    cleaned = _clean_control_chars(raw)
    
    try:
        logger.info("Thử parse JSON như một mảng...")
        data = json.loads(cleaned)
        if isinstance(data, dict):
            data = data.get('data', [data])
        df = pd.DataFrame(data)
        logger.info(f"Parse JSON thành công, kích thước: {df.shape}")
        return df
    except json.JSONDecodeError as e:
        logger.warning(f"Parse JSON thất bại: {e}, thử JSON Lines...")
    
    rows = []
    for line in cleaned.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            try:
                rows.append(json.loads(_clean_control_chars(line)))
            except Exception:
                continue
    
    if not rows:
        logger.error("Không thể parse JSON. Kiểm tra định dạng file.")
        raise ValueError("Không thể parse JSON. Kiểm tra định dạng file.")
    
    logger.info(f"Parse JSON Lines thành công, {len(rows)} records")
    return pd.DataFrame(rows)

df_raw = load_reviews(DATA_PATH)
logger.info(f"Tải dữ liệu hoàn tất: {df_raw.shape}")
print("Kích thước raw:", df_raw.shape)

In [ ]:
def clean_and_prepare_data(df, *, use_underthesea: bool = False, stopwords_path: str | None = None, min_words: int = 5):
    logger.info("Bắt đầu làm sạch và chuẩn bị dữ liệu...")
    df = df.copy()
    
    if 'created_at' in df.columns:
        logger.info("Chuyển đổi cột thời gian...")
        df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')

    if 'rating' in df.columns:
        logger.info("Tạo nhãn sentiment 3 lớp từ rating...")
        df['sentiment_3c'] = df['rating'].apply(label_from_rating_3class)

    text_cols = [c for c in ['title','content'] if c in df.columns]
    logger.info(f"Ghép văn bản từ các cột: {text_cols}")
    df['text'] = df.apply(lambda r: safe_join_text(r, text_cols), axis=1) if text_cols else ''

    original_size = len(df)
    logger.info(f"Kích thước ban đầu: {original_size}")
    
    logger.info("Loại bỏ dữ liệu thiếu...")
    df = df.dropna(subset=['text','rating','sentiment_3c'])
    df = df[df['text'].str.len() > 0].reset_index(drop=True)
    logger.info(f"Sau loại bỏ dữ liệu thiếu: {len(df)}")

    logger.info("Tải stopwords và tạo tokenizer...")
    stopwords = _load_stopwords(STOPWORDS_PATH)
    tokenizer = make_vn_tokenizer(use_underthesea, stopwords)
    
    logger.info("Tokenize văn bản...")
    df['tokens'] = df['text'].apply(lambda s: tokenizer(s))
    
    logger.info(f"Lọc reviews có ít nhất {min_words} từ...")
    df = df[df['tokens'].apply(lambda ts: len(ts) >= int(min_words))].reset_index(drop=True)
    logger.info(f"Sau lọc độ dài: {len(df)}")

    logger.info("Tạo processed_text và loại trùng...")
    df['processed_text'] = df['tokens'].apply(lambda ts: ' '.join(ts))
    before_dedup = len(df)
    df = df.drop_duplicates(subset=['processed_text']).reset_index(drop=True)
    logger.info(f"Loại bỏ {before_dedup - len(df)} bản ghi trùng lặp")

    logger.info(f"Hoàn thành làm sạch: {original_size} -> {len(df)}")
    print(f"Kích thước ban đầu: {original_size}")
    print(f"Sau làm sạch: {len(df)}")
    return df

# === 2.3. Kết quả xử lý ===
df = clean_and_prepare_data(df_raw, use_underthesea=USE_UNDERTHESEA, stopwords_path=STOPWORDS_PATH, min_words=MIN_WORDS)
# So sánh nội dung 1-2 samples trước/sau xử lý
# Chỉnh sửa danh sách chỉ số theo ý bạn, ví dụ: [10] hoặc [10, 123]
sample_idxs = [0, 1]
_text_cols = [c for c in ['title','content'] if c in df_raw.columns]
_tok_fn = make_vn_tokenizer(USE_UNDERTHESEA, _load_stopwords(STOPWORDS_PATH))
print("--------------------------------")
print("So sánh dữ liệu trước và sau xử lý:")
for i in sample_idxs:
    if i < 0 or i >= len(df_raw):
        print(f"Bỏ qua index {i} (ngoài phạm vi)")
        continue
    raw_row = df_raw.iloc[i]
    raw_text = safe_join_text(raw_row, _text_cols) if _text_cols else str(raw_row.get('content', ''))
    tokens = _tok_fn(raw_text)
    processed = ' '.join(tokens)
    print(f"\n=== Sample index {i} ===")
    print("Raw:", raw_text[:500])
    print("Processed:", processed[:500])
    print(f"Tokens ({len(tokens)}):", tokens[:50])
df.head(2)

# 3. Khám phá và trực quan hóa dữ liệu (30%)


### 3.1. Phân phối rating

**Mục tiêu**: Phân tích phân phối của rating trong dataset

**Visualizations**:
- Biểu đồ cột phân phối rating (1-5 sao) với màu sắc theo sentiment
- Thống kê số lượng và tỷ lệ phần trăm

### 3.2. Phân tích độ dài review

**Mục tiêu**: Khám phá độ dài của các review

**Phân tích**:
- Histogram phân phối độ dài theo số từ
- Thống kê mô tả: mean, median, std, min, max
- So sánh độ dài review giữa các lớp sentiment

### 3.3. Phân tích từ vựng

**Mục tiêu**: Khám phá từ vựng phổ biến và đặc trưng

**Phân tích**:
- Top 30 từ phổ biến nhất trong toàn bộ dataset
- Word Cloud visualization cho từ phổ biến
- Top 10 từ đặc trưng cho từng lớp sentiment (negative/neutral/positive)
- Phân tích cụm từ phủ định và tần suất xuất hiện



In [ ]:
# Giảm số mẫu để cân bằng data
label_counts = df['sentiment_3c'].value_counts()
min_count = label_counts.min()
print(f"Phân phối ban đầu:")
print(label_counts)
print(f"\nResampling mỗi lớp xuống còn {min_count} mẫu (lớp nhỏ nhất)")

balanced_df = (
    df.groupby("sentiment_3c", group_keys=False)
    .apply(lambda grp: grp.sample(n=min_count, random_state=42))
    .reset_index(drop=True)
)

print("\nPhân phối sau khi cân bằng:")
print(balanced_df["sentiment_3c"].value_counts())
print(f"\nTổng số mẫu sau cân bằng: {len(balanced_df)}")


In [ ]:
# Tách từ và xử lý văn bản tiếng Việt
# LƯU Ý: Sử dụng balanced_df (dữ liệu đã cân bằng) cho tất cả các phân tích
logger.info("Bắt đầu phân tích khám phá dữ liệu (EDA) với dữ liệu đã cân bằng...")

# === 3.1 Phân phối rating ===
logger.info("Phân tích phân phối rating...")
rating_counts = balanced_df['rating'].value_counts().sort_index()
print("Phân phối rating (từ balanced_df):\n", rating_counts)

# Màu theo mapping cảm xúc: 1-2=negative (đỏ), 3=neutral (xám), 4-5=positive (xanh)
star_to_color = {1: '#E74C3C', 2: '#E74C3C', 3: '#95A5A6', 4: '#2ECC71', 5: '#2ECC71'}
bar_colors = [star_to_color.get(int(star), '#95A5A6') for star in rating_counts.index]

plt.figure(figsize=(10, 6))
rating_counts.plot(kind='bar', color=bar_colors)
plt.title('Phân phối Rating', fontsize=14, fontweight='bold')
plt.xlabel('Rating (sao)')
plt.ylabel('Số lượng review')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
# Thêm chú giải màu
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#E74C3C', label='Negative (1-2 sao)'),
    Patch(facecolor='#95A5A6', label='Neutral (3 sao)'),
    Patch(facecolor='#2ECC71', label='Positive (4-5 sao)')
]
plt.legend(handles=legend_elements, title='Sentiment', loc='upper right')
plt.tight_layout()
plt.show()

# === 3.2. Phân tích độ dài review ===
logger.info("Phân tích độ dài review...")
balanced_df['char_len'] = balanced_df['text'].str.len()
balanced_df['word_len'] = balanced_df['tokens'].apply(len)

print("\nThống kê độ dài:")
display(balanced_df[['char_len','word_len']].describe())

plt.figure(figsize=(8, 5))
balanced_df['word_len'].hist(bins=50, alpha=0.8, color='#3498DB')
plt.title('Phân phối độ dài (từ) - Dữ liệu đã cân bằng')
plt.xlabel('Số từ'); plt.ylabel('Số review'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# === 3.2. Phân tích từ vựng ===
logger.info("Phân tích từ phổ biến (từ balanced_df)...")
import collections
counter_all = collections.Counter()
for toks in balanced_df['tokens']:
    counter_all.update(toks)

top_words = counter_all.most_common(30)
print("\nTop 30 từ phổ biến:")
for w,c in top_words:
    print(f"{w}: {c}")

# Tạo Word Cloud cho top từ phổ biến
try:
    from wordcloud import WordCloud
    _WORDCLOUD_AVAILABLE = True
except ImportError:
    _WORDCLOUD_AVAILABLE = False
    print("Cài đặt wordcloud: pip install wordcloud")

if _WORDCLOUD_AVAILABLE:
    # Tạo từ điển tần suất từ top_words
    word_freq = dict(top_words[:50])  # Lấy top 50 từ
    
    # Tạo word cloud
    wordcloud = WordCloud(
        width=800, height=400,
        background_color='white',
        max_words=50,
        colormap='viridis',
        font_path=None,  # Sử dụng font mặc định
        relative_scaling=0.5,
        random_state=42
    ).generate_from_frequencies(word_freq)
    
    plt.figure(figsize=(12, 6))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('Word Cloud - Top từ phổ biến', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()
else:
    # Fallback: hiển thị danh sách nếu không có wordcloud
    print("Top 20 từ phổ biến (không có wordcloud):")
    for i, (word, count) in enumerate(top_words[:20], 1):
        print(f"{i:2d}. {word}: {count}")

# Top từ theo sentiment
# LƯU Ý: Sử dụng balanced_df (dữ liệu đã cân bằng)
logger.info("Phân tích từ theo sentiment (từ balanced_df)...")

for sentiment in ['negative','neutral','positive']:
    subset = balanced_df[balanced_df['sentiment_3c'] == sentiment]
    counter_sentiment = collections.Counter()
    for toks in subset['tokens']:
        counter_sentiment.update(toks)
    print(f"\nTop 10 từ trong {sentiment}:")
    for w,c in counter_sentiment.most_common(10):
        print(f"  {w}: {c}")

logger.info("Hoàn thành phân tích khám phá dữ liệu")



# 4. Phân tích và mô hình hóa (25%)

### Kiến trúc mô hình

### 4.1. Pipeline xử lý
- **TF-IDF Vectorizer**: 
  - N-gram range: (1,2) - bao gồm unigram và bigram
  - Max features: 40,000 từ phổ biến nhất
  - Min/Max document frequency: 2-95%
- **Logistic Regression**:
  - Class weight: 'balanced' để xử lý imbalanced data
  - Solver: 'lbfgs' cho convergence tốt
  - Max iterations: 1000

### 4.2. Huấn luyện mô hình
- **Chia dữ liệu**: 80% train, 20% test với stratified sampling
- **Cross-validation**: Đảm bảo phân phối đều giữa các lớp
- **Feature engineering**: Sử dụng processed_text đã được tokenize

### 4.3. Đánh giá mô hình
- **Classification Report**: Precision, Recall, F1-score cho từng lớp
- **Confusion Matrix**: Ma trận nhầm lẫn với visualization
- **Demo dự đoán**: Test mô hình với các câu mẫu thực tế


In [ ]:
# === 4.2. Huấn luyện mô hình ===
# LƯU Ý: Sử dụng balanced_df (dữ liệu đã cân bằng) cho train/test
logger.info("Bắt đầu xây dựng và huấn luyện mô hình với dữ liệu đã cân bằng...")

# Chuẩn bị dữ liệu train/test
logger.info("Chuẩn bị dữ liệu train/test từ balanced_df...")
df_clean = balanced_df.dropna(subset=['processed_text','sentiment_3c']).reset_index(drop=True)
X = df_clean['processed_text'].astype(str).values
y = df_clean['sentiment_3c'].astype(str).values

print(f"Kích thước dữ liệu: {len(X)}")
unique, counts = np.unique(y, return_counts=True)
print("Phân phối nhãn:")
for label, count in zip(unique, counts):
    print(f"  {label}: {count} ({count/len(y)*100:.1f}%)")

logger.info("Chia dữ liệu train/test (80/20)...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
logger.info(f"Train set: {len(X_train)}, Test set: {len(X_test)}")

# Tạo pipeline mô hình
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=40000,
        ngram_range=(1,2),
        lowercase=False,
        min_df=2,
        max_df=0.95,
        tokenizer=str.split,
        token_pattern=None
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        solver='lbfgs',
        random_state=42
    ))
])

# Huấn luyện mô hình
logger.info("Bắt đầu huấn luyện mô hình...")
print("\nHuấn luyện mô hình...")
model = pipe.fit(X_train, y_train)
logger.info("Huấn luyện mô hình hoàn tất")

print("Mô hình đã được huấn luyện thành công!")


In [ ]:
# 4.3) Đánh giá mô hình
logger.info("Bắt đầu đánh giá mô hình...")

# Dự đoán trên test set
logger.info("Dự đoán trên test set...")
y_pred = model.predict(X_test)

# Classification report
print("\n### Classification Report")
print(classification_report(y_test, y_pred, digits=4))

# Confusion Matrix
labels_order = ['negative','neutral','positive']
cm = confusion_matrix(y_test, y_pred, labels=labels_order)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_order)
disp.plot(values_format='d', cmap='Blues')
plt.title("Confusion Matrix - 3 lớp")
plt.gcf().set_size_inches(7,5)
plt.tight_layout()
plt.show()

# Demo dự đoán
logger.info("Thực hiện demo dự đoán...")
demo_samples = [
    "Sản phẩm rất tốt, chất lượng cao, giao hàng nhanh. Tôi rất hài lòng!",
    "Hàng tệ quá, không giống mô tả, giao hàng chậm. Thất vọng!",
    "Sản phẩm bình thường, không tệ nhưng cũng không xuất sắc.",
    "Sản phẩm tạm được, nhưng shop giao hàng chậm chạp, ko uy tín"
]

tok = make_vn_tokenizer(USE_UNDERTHESEA, _load_stopwords(STOPWORDS_PATH))
demo_tok = [' '.join(tok(s)) for s in demo_samples]
preds = model.predict(demo_tok)
probas = model.predict_proba(demo_tok)
clf = model.named_steps['clf']

print("\n### Demo Dự đoán")
for s, p, pr in zip(demo_samples, preds, probas):
    print(f"\nText: {s}")
    print(f"Pred: {p}")
    print(f"Prob: {dict(zip(clf.classes_, pr))}")

logger.info("Hoàn thành đánh giá mô hình")



## Kết luận và đề xuất (10%)

### Tổng kết kết quả

**Điểm nổi bật**:
- Pipeline tiền xử lý hiệu quả: loại nhiễu (URL/email/emoji), tách từ tiếng Việt, ghép cụm phủ định
- Mô hình **TF-IDF + LogisticRegression** đạt hiệu suất tốt cho bài toán phân loại 3 lớp sentiment
- Xử lý stopwords và cụm phủ định giúp cải thiện chất lượng features

### Đề xuất cải thiện

**1. Cải thiện dữ liệu**:
- Bổ sung và tinh chỉnh danh sách stopwords theo miền dữ liệu
- Cân bằng dữ liệu giữa các lớp (SMOTE, undersampling)
- Thu thập thêm dữ liệu cho lớp thiểu số

**2. Cải thiện mô hình**:
- Thử nghiệm PhoBERT hoặc mô hình transformer cho tiếng Việt
- So sánh với các tokenizer khác: VnCoreNLP, PyVi
- Ensemble methods: voting, stacking
- Hyperparameter tuning với GridSearch/RandomSearch

**3. Triển khai thực tế**:
- Xây dựng REST API cho dự đoán real-time
- Monitoring: theo dõi data drift và model performance
- A/B testing và retrain định kỳ
- Tối ưu inference time cho production

**Logging:**
- Tất cả quá trình đã được ghi log vào file `sentiment_analysis.log`
- Log bao gồm: thông tin khởi tạo, tải dữ liệu, làm sạch, EDA, huấn luyện mô hình
- Có thể theo dõi tiến trình và debug khi cần thiết
